In [ ]:

#Loads the ml-latest-small dataset (movies.csv, ratings.csv, tags.csv)

import os
import pandas as pd

DATA_DIR = "/content/"


def load_ratings() -> pd.DataFrame:
    return pd.read_csv(os.path.join(DATA_DIR, "ratings.csv"))


def load_movies() -> pd.DataFrame:
    df = pd.read_csv(os.path.join(DATA_DIR, "movies.csv"))
    df["genres"] = df["genres"].replace("(no genres listed)", "")
    df["genres_text"] = df["genres"].str.replace("|", " ", regex=False)
    return df


def load_tags() -> pd.DataFrame:
    return pd.read_csv(os.path.join(DATA_DIR, "tags.csv"))


def load_links() -> pd.DataFrame:
    return pd.read_csv(os.path.join(DATA_DIR, "links.csv"))


def build_movie_content(movies: pd.DataFrame, tags: pd.DataFrame) -> pd.DataFrame:
    """Combine genres + all user tags per movie into one text field for content-based similarity."""
    tags_agg = (
        tags.groupby("movieId")["tag"]
        .apply(lambda x: " ".join(x.astype(str)))
        .rename("tags_text")
    )
    movies = movies.merge(tags_agg, on="movieId", how="left")
    movies["tags_text"] = movies["tags_text"].fillna("")
    movies["content_text"] = (movies["genres_text"] + " " + movies["tags_text"]).str.lower()
    return movies


if __name__ == "__main__":
    ratings = load_ratings()
    movies = load_movies()
    tags = load_tags()
    movies = build_movie_content(movies, tags)
    print(movies[["movieId", "title", "content_text"]].head())
    print(f"Users: {ratings.userId.nunique()}, Movies: {ratings.movieId.nunique()}, Ratings: {len(ratings)}")

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        content_text  
0  adventure animation children comedy fantasy pi...  
1  adventure children fantasy fantasy magic board...  
2                           comedy romance moldy old  
3                              comedy drama romance   
4                            comedy pregnancy remake  
Users: 610, Movies: 9724, Ratings: 100836
